In [1]:
%%bash
set -e
pip install -q deepxde tensorflow-probability tf-keras 2>&1 | tail -2
rm -rf /tmp/repo
git clone -q https://github.com/Mathewboat/Pinn-heat-fluid.git /tmp/repo
cd /tmp/repo
git log -1 --format="commit: %h %s"
ls experiments
python - <<'PY'
import tensorflow as tf, deepxde as dde
n = len(tf.config.list_physical_devices("GPU"))
print("TF", tf.__version__, "| DeepXDE", dde.__version__, "| GPUs:", n)
assert n >= 1, "No GPU visible: turn the GPU accelerator on in Session options"
PY
nproc

commit: 7fde41e Recipe experiment: XLA off by default (hard constraints pathologically slow with XLA)
00_env_benchmark.py
01_heat1d_recipe.py
run_grid.py
Setting the default backend to "tensorflow". You can change it in the ~/.deepxde/config.json file or export the DDE_BACKEND environment variable. Valid options are: tensorflow.compat.v1, tensorflow, pytorch, jax, paddle (all lowercase)
TF 2.20.0 | DeepXDE 1.15.0 | GPUs: 2
4


No backend selected.
Finding available backend...
Found tensorflow
Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.
Enable just-in-time compilation with XLA.



In [2]:
%%bash
set -e
cd /tmp/repo
timeout 900 python -u experiments/01_heat1d_recipe.py --precision float32 --constraint hard --adam-iters 500 --lbfgs-maxiter 100 --outdir /kaggle/working/results/smoke

Set the default float type to float32
Compiling model...
'compile' took 0.546198 s

Training model...

Step      Train loss    Test loss     Test metric   
0         [6.35e+00]    [6.42e+00]    [2.10e+00]    
125       [4.34e-01]    [4.13e-01]    [3.38e-01]    
250       [2.75e-02]    [2.38e-02]    [5.68e-02]    
375       [9.51e-03]    [7.77e-03]    [2.59e-02]    
500       [4.58e-03]    [3.58e-03]    [1.53e-02]    

Best model at step 500:
  train loss: 4.58e-03
  test loss: 3.58e-03
  test metric: [1.53e-02]

'train' took 9.408599 s

Compiling model...
'compile' took 0.009850 s

Training model...

Step      Train loss    Test loss     Test metric   
500       [4.58e-03]    [3.58e-03]    [1.53e-02]    
600       [1.14e-05]    [1.05e-05]    [3.76e-04]    

Best model at step 600:
  train loss: 1.14e-05
  test loss: 1.05e-05
  test metric: [3.76e-04]

'train' took 41.037076 s

heat1d_float32_hard_s0: after Adam L2=1.48e-02 | final L2=3.61e-04 flux=1.15e-03 energy=2.08e-04 | L-BFGS 100 

Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.
Enable just-in-time compilation with XLA.

Disable just-in-time compilation with XLA.
I0000 00:00:1793147631.746578      97 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793147631.750960      97 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [3]:
%%bash
test -d /tmp/repo || { echo "Run Cell 1 first (/tmp/repo is missing)"; exit 1; }
cd /tmp/repo
python experiments/run_grid.py --script experiments/01_heat1d_recipe.py \
  --grid precision=float32,float64 constraint=soft,hard \
  --seeds 0 1 2 3 4 --gpus 0 1 --procs-per-gpu 2 \
  --name-template "heat1d_{precision}_{constraint}_s{seed}" \
  --outdir /kaggle/working/results/heat1d_recipe \
  --extra --lbfgs-maxiter 3000 || echo "Some jobs failed: see results/heat1d_recipe/logs"

20 jobs total, 0 already done, 20 to run
[     0s] start  heat1d_float32_soft_s0  (gpu 0)
[     0s] start  heat1d_float32_soft_s1  (gpu 0)
[     0s] start  heat1d_float32_soft_s2  (gpu 1)
[     0s] start  heat1d_float32_soft_s3  (gpu 1)
[   410s] done  heat1d_float32_soft_s3  (410s)  [1 finished]
[   412s] start  heat1d_float32_soft_s4  (gpu 1)
[   468s] done  heat1d_float32_soft_s0  (468s)  [2 finished]
[   470s] start  heat1d_float32_hard_s0  (gpu 0)
[   490s] done  heat1d_float32_soft_s1  (490s)  [3 finished]
[   490s] done  heat1d_float32_soft_s2  (490s)  [4 finished]
[   492s] start  heat1d_float32_hard_s1  (gpu 0)
[   492s] start  heat1d_float32_hard_s2  (gpu 1)
[   850s] done  heat1d_float32_soft_s4  (438s)  [5 finished]
[   852s] start  heat1d_float32_hard_s3  (gpu 1)
[   892s] done  heat1d_float32_hard_s0  (422s)  [6 finished]
[   894s] start  heat1d_float32_hard_s4  (gpu 0)
[   928s] done  heat1d_float32_hard_s1  (436s)  [7 finished]
[   930s] start  heat1d_float64_soft_s0  (

In [4]:
import json, glob, numpy as np
from collections import defaultdict
RESULTS = "/kaggle/working/results/heat1d_recipe"
groups = defaultdict(list)
for f in glob.glob(RESULTS + "/*.json"):
    d = json.load(open(f)); groups[(d["precision"], d["constraint"])].append(d)
if not groups:
    print("No result files found: the grid did not produce any JSON. Check the logs folder.")
else:
    print(f"{'precision':9s} {'constraint':10s} {'n':>2s} | {'L2 median':>10s} {'min':>9s} {'max':>9s} | "
          f"{'flux L2':>9s} {'energy L2':>10s} | {'L-BFGS its':>10s} {'early stop':>10s} | {'min/run':>7s}")
    for (prec, con), runs in sorted(groups.items()):
        g = lambda k: np.array([r["final"][k] for r in runs])
        its = np.array([r["lbfgs_iters_run"] for r in runs])
        mins = np.array([(r["seconds_adam"] + r["seconds_lbfgs"]) / 60 for r in runs])
        print(f"{prec:9s} {con:10s} {len(runs):>2d} | {np.median(g('l2')):10.2e} {g('l2').min():9.2e} {g('l2').max():9.2e} | "
              f"{np.median(g('flux_l2')):9.2e} {np.median(g('energy_l2')):10.2e} | {int(np.median(its)):>10d} "
              f"{sum(r['lbfgs_stopped_early'] for r in runs):>4d}/{len(runs):<5d} | {mins.mean():7.1f}")

precision constraint  n |  L2 median       min       max |   flux L2  energy L2 | L-BFGS its early stop | min/run
float32   hard        5 |   1.63e-05  9.56e-06  2.32e-05 |  4.04e-05   8.58e-06 |        378    5/5     |     7.1
float32   soft        5 |   2.99e-04  2.72e-04  4.27e-04 |  7.47e-04   1.71e-04 |        449    5/5     |     7.4
float64   hard        5 |   1.62e-06  9.34e-07  3.11e-06 |  5.07e-06   8.83e-07 |       3000    0/5     |    31.6
float64   soft        5 |   4.81e-05  2.58e-05  1.02e-04 |  1.56e-04   3.03e-05 |       3000    0/5     |    37.6


In [5]:
%%bash
cd /kaggle/working
if [ -d results/heat1d_recipe ]; then
  echo "JSON files found: $(ls results/heat1d_recipe/*.json 2>/dev/null | wc -l) of 20"
  zip -qr heat1d_recipe.zip results && ls -l heat1d_recipe.zip
else
  echo "No results folder: the grid cell has not run in this session."
fi

JSON files found: 20 of 20
-rw-r--r-- 1 root root 37806 Oct 28 02:31 heat1d_recipe.zip
